In [1]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random
from transformers import AutoTokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM, GPTNeoConfig, GPTNeoForCausalLM

cfg_param = "8M"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
epochs = 1
seed = 3407
batch_size = 64
window_size = 256
lr = 1e-3

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
    

tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
model = AutoModelForCausalLM.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")

# Initializing a model (with random weights) from the EleutherAI/gpt-neo-1.3B style configuration
model = GPTNeoForCausalLM(model.config)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in model: {num_params}")

/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of parameters in model: 19702528


In [2]:
# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace")
else:
    print("Warning: HF_TOKEN not found in .env file")

# Set your HuggingFace username/organization
HF_USERNAME = os.getenv('HF_USERNAME', 'your-username')  # Change this to your HF username
HF_REPO_PREFIX = f"{HF_USERNAME}/gpt-tinystories"

# Global variable to track data indices used since last checkpoint
data_tracker = {
    'train_indices': [],
    'train_data': [],
    'val_indices': [],
    'val_data': []
}

def reset_data_tracker():
    """Reset the data tracker for a new checkpoint period"""
    global data_tracker
    data_tracker = {
        'train_indices': [],
        'train_data': [],
        'val_indices': [],
        'val_data': []
    }

def save_checkpoint(model, optimizer, updates, checkpoint_name, data_tracker_state=None):
    """
    Save checkpoint to HuggingFace Hub
    Args:
        model: The model to save
        optimizer: The optimizer state to save
        updates: Number of training updates/steps
        checkpoint_name: Name for this checkpoint (e.g., "checkpoint-1000")
        data_tracker_state: Dictionary containing train/val indices and data used since last checkpoint
    """
    # Get the base model if wrapped in DataParallel
    model_to_save = model.module if hasattr(model, 'module') else model
    
    # Create a temporary directory to save checkpoint
    temp_dir = f"temp_checkpoint_{updates}"
    os.makedirs(temp_dir, exist_ok=True)
    
    # Save model
    model_to_save.save_pretrained(temp_dir)
    
    # Save optimizer state and training info
    checkpoint_dict = {
        'optimizer_state_dict': optimizer.state_dict(),
        'updates': updates,
    }
    training_state_filename = f'checkpoint_{updates}.pt'
    torch.save(checkpoint_dict, os.path.join(temp_dir, training_state_filename))
    
    # Save data tracker if provided
    if data_tracker_state is not None:
        data_tracker_filename = f'data_tracker_{updates}.json'
        with open(os.path.join(temp_dir, data_tracker_filename), 'w') as f:
            json.dump(data_tracker_state, f, indent=2)
    
    # Push to HuggingFace Hub
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    commit_message = f"Checkpoint at step {updates}"
    
    try:
        model_to_save.push_to_hub(
            repo_name,
            commit_message=commit_message,
            private=True,  # Set to False if you want public repo
        )
        
        # Upload training state separately
        api = HfApi()
        api.upload_file(
            path_or_fileobj=os.path.join(temp_dir, training_state_filename),
            path_in_repo=training_state_filename,
            repo_id=repo_name,
            commit_message=f"Update training state at step {updates}",
        )
        
        # Upload data tracker if it exists
        if data_tracker_state is not None:
            api.upload_file(
                path_or_fileobj=os.path.join(temp_dir, data_tracker_filename),
                path_in_repo=data_tracker_filename,
                repo_id=repo_name,
                commit_message=f"Update data tracker at step {updates}",
            )
        
        print(f"Checkpoint saved to HuggingFace: {repo_name}")
        logging.info(f"Checkpoint saved to HuggingFace: {repo_name} at step {updates}")
        if data_tracker_state is not None:
            print(f"Saved {len(data_tracker_state['train_data'])} training samples, {len(data_tracker_state['val_data'])} validation samples")
            logging.info(f"Saved {len(data_tracker_state['train_data'])} training samples, {len(data_tracker_state['val_data'])} validation samples")
    except Exception as e:
        print(f"Error saving checkpoint to HuggingFace: {e}")
        logging.error(f"Error saving checkpoint to HuggingFace: {e}")
    finally:
        # Clean up temporary directory
        import shutil
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

def load_checkpoint(model, optimizer, checkpoint_name):
    """
    Load checkpoint from HuggingFace Hub
    Args:
        model: The model to load weights into
        optimizer: The optimizer to load state into
        checkpoint_name: Name of the checkpoint to load
    Returns:
        updates: Number of training updates/steps from checkpoint
    """
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    
    try:
        # Get the base model if wrapped in DataParallel
        model_to_load = model.module if hasattr(model, 'module') else model
        
        # Load model from HuggingFace Hub
        loaded_model = GPTNeoForCausalLM.from_pretrained(repo_name)
        model_to_load.load_state_dict(loaded_model.state_dict())
        
        # Download and load training state - get the latest checkpoint file
        from huggingface_hub import hf_hub_download, list_repo_files
        
        # List all files in the repo and find the latest checkpoint
        files = list_repo_files(repo_id=repo_name)
        checkpoint_files = [f for f in files if f.startswith('checkpoint_') and f.endswith('.pt')]
        
        if checkpoint_files:
            # Sort by checkpoint number and get the latest
            checkpoint_files.sort(key=lambda x: int(x.split('_')[1].split('.')[0]), reverse=True)
            latest_checkpoint = checkpoint_files[0]
            
            training_state_path = hf_hub_download(repo_id=repo_name, filename=latest_checkpoint)
            checkpoint_dict = torch.load(training_state_path)
            
            optimizer.load_state_dict(checkpoint_dict['optimizer_state_dict'])
            updates = checkpoint_dict['updates']
            
            print(f"Checkpoint loaded from HuggingFace: {repo_name} (step {updates})")
            logging.info(f"Checkpoint loaded from HuggingFace: {repo_name} at step {updates}")
            
            return updates
        else:
            print(f"No checkpoint files found in {repo_name}")
            return 0
    except Exception as e:
        print(f"Error loading checkpoint from HuggingFace: {e}")
        logging.error(f"Error loading checkpoint from HuggingFace: {e}")
        return 0

class TrackingDataset(Dataset):
    """
    Wrapper dataset that tracks which samples are accessed
    """
    def __init__(self, dataset, mode='train'):
        self.dataset = dataset
        self.mode = mode
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        # Track this access
        global data_tracker
        if self.mode == 'train':
            data_tracker['train_indices'].append(idx)
            data_tracker['train_data'].append({
                'index': idx,
                'text': self.dataset[idx]['text']
            })
        else:
            data_tracker['val_indices'].append(idx)
            data_tracker['val_data'].append({
                'index': idx,
                'text': self.dataset[idx]['text']
            })
        
        return self.dataset[idx]

print("Checkpoint functions loaded")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to HuggingFace
Checkpoint functions loaded


In [ ]:
import random
# Set up logger
current_time = datetime.now().strftime("%m%d_%H%M%S")
import os
if not os.path.exists("logs"):
    os.makedirs("logs")

log_filename = f"logs/training_{cfg_param}_{current_time}.log"
logging.basicConfig(filename=log_filename, level=logging.INFO,
                    format='%(asctime)s %(levelname)s: %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S')

# Load dataset and tokenizer
model_name = 'roneneldan/TinyStories'
dataset = load_dataset(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Wrap datasets with tracking
train_dataset = TrackingDataset(dataset['train'], mode='train')
valid_dataset = TrackingDataset(dataset['validation'], mode='val')

# Create deterministic generators for shuffling
train_generator = torch.Generator()
train_generator.manual_seed(seed)
valid_generator = torch.Generator()
valid_generator.manual_seed(seed)

# Instantiate dataloader with deterministic shuffling
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    generator=train_generator,
    worker_init_fn=lambda worker_id: np.random.seed(seed + worker_id)
)
for batch in train_loader:
    print(batch['text'][0])
    break

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    generator=valid_generator,
    worker_init_fn=lambda worker_id: np.random.seed(seed + worker_id)
)

# Instantiate model and optimizer

def estimate_loss(model, tokenizer, valid_loader, device='cuda'):
    model.eval()
    with torch.no_grad():
        losses = torch.zeros(40)
        for k,batch in enumerate(valid_loader):
            tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length = 256, truncation = True)['input_ids'].to(device)
            outputs = model(tokenized, labels=tokenized)
            loss = outputs.loss
            if torch.cuda.device_count() > 1:
                loss = loss.mean()
            losses[k] = loss.item()
            if k == 40 - 1 :
                break
    model.train()
    return losses.mean()


if torch.cuda.device_count() > 1:
    # if multiple gpus on single machine
    model = nn.DataParallel(model)
model.to(device)
optim = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.95))

updates = 0
hf_repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
resume_training = False
if resume_training:
    logging.info(f"Resuming training from {hf_repo_name}")
    updates = load_checkpoint(model, optim, hf_repo_name)

# Setup weights & biases
run = wandb.init(
    project="gpt-tinystories",
    name=f"gpt-tinystories-{cfg_param}-{current_time}",
    config={
        "cfg_param": cfg_param,
        "learning_rate": lr,
        "batch_size": batch_size,
        "hf_repo": hf_repo_name,
        "log_filename": log_filename,
        "seed": seed,
        "epochs": epochs
    },
)
logging.info(f"cfg_param: {cfg_param}, lr: {lr}, batch_size: {batch_size}, hf_repo: {hf_repo_name}, log_filename: {log_filename}, seed: {seed}, epochs: {epochs}")

# Reset data tracker at start
reset_data_tracker()

# Training loop
for epoch in range(epochs):
    logging.info(f"Epoch: {epoch+1}")
    for batch in tqdm(train_loader):
        optim.zero_grad()
        tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length=256, truncation=True)['input_ids'].to(device)
        outputs = model(tokenized, labels=tokenized)
        loss = outputs.loss
        if torch.cuda.device_count() > 1:
            loss = loss.mean()
        loss.backward()
        optim.step()
        updates += 1
        if updates % 200 == 0:
            validation_loss = estimate_loss(model, tokenizer, valid_loader)
            tqdm.write(f"Train_{epoch+1}_{updates}: {validation_loss}")
            logging.info(f"Train_{epoch+1}_{updates}: {validation_loss}")
            wandb.log({"train_loss": loss, "val_loss": validation_loss})
        if updates % 1000 == 0:
            # Save checkpoint with data tracker
            save_checkpoint(model, optim, updates, f"checkpoint-{updates}", data_tracker_state=data_tracker)
            # Reset tracker after saving
            reset_data_tracker()
    logging.info("TRAINING COMPLETE")
    logging.info("Computing final validation loss..")
    # Validation loop
    with torch.no_grad():
        loss_valid = 0
        for batch in tqdm(valid_loader):
            tokenized = tokenizer(batch['text'], padding=True, return_tensors='pt', max_length=512,truncation=True)['input_ids'].to(device)
            outputs = model(tokenized, labels=tokenized)
            loss = outputs.loss
            loss_valid += loss.mean().item()
        logging.info(f"Final validation loss: {loss_valid / len(valid_loader)}")
        save_checkpoint(model, optim, updates, f"checkpoint-final-{updates}", data_tracker_state=data_tracker)
        
        # Log HuggingFace repo to wandb
        wandb.log({"hf_repo": hf_repo_name})
        print(f"Final model saved to HuggingFace: {hf_repo_name}")

# Trained model output

Once upon a time, there was a sweet little dog named Spot. Spot loved to play with his ball. One day, he saw a big tree in the park. Spot had an urge to play near the tree. He did not know why, but he felt like something fun would happen there.

Spot ran to the tree and started to play with his ball. He saw a pretty bird up in the tree. He wanted to be friends with the bird. Spot tried to point at the bird with his paw, but the bird did not see him. Spot felt sad, but he did not give up. He knew there was a way to make the bird see him.

The next day, Spot went back to the tree. He had a plan. He found a long stick and held it in his mouth. Spot used the stick to point at the bird. This time, the bird saw him! The bird flew down and played with Spot and his ball. They became the best of friends, and Spot was happy. His urge to play near the tree had led him to a new friend.


wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  1%|          | 201/33121 [00:18<3:07:26,  2.93it/s]

Train_1_200: 3.2659599781036377


  1%|          | 401/33121 [00:36<3:11:41,  2.84it/s]

Train_1_400: 2.7257885932922363


  1%|▏         | 421/33121 [00:38<48:58, 11.13it/s]  

In [ ]:
# Helper function to load and inspect saved training data from a checkpoint
def load_checkpoint_data(checkpoint_step):
    """
    Load the training/validation data used for a specific checkpoint
    Args:
        checkpoint_step: The checkpoint step number (e.g., 1000, 2000)
    Returns:
        Dictionary with train_data and val_data
    """
    from huggingface_hub import hf_hub_download
    import json
    
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    data_tracker_filename = f'data_tracker_{checkpoint_step}.json'
    
    try:
        # Download the data tracker file
        data_path = hf_hub_download(repo_id=repo_name, filename=data_tracker_filename)
        
        with open(data_path, 'r') as f:
            data_tracker_loaded = json.load(f)
        
        print(f"Loaded data tracker for checkpoint {checkpoint_step}")
        print(f"  Training samples: {len(data_tracker_loaded['train_data'])}")
        print(f"  Validation samples: {len(data_tracker_loaded['val_data'])}")
        print(f"  Unique training indices: {len(set(data_tracker_loaded['train_indices']))}")
        print(f"  Unique validation indices: {len(set(data_tracker_loaded['val_indices']))}")
        
        return data_tracker_loaded
    except Exception as e:
        print(f"Error loading checkpoint data: {e}")
        return None

# Example usage (uncomment to use):
data = load_checkpoint_data(1000)
if data:
    # Access first training sample
    print("First training sample:", data['train_data'][0])
    
    # Get all training texts
    train_texts = [sample['text'] for sample in data['train_data']]
    
    # Verify reproducibility - check if indices match expected order
    print("Training indices:", data['train_indices'][:10])

Loaded data tracker for checkpoint 1000
  Training samples: 64000
  Validation samples: 12800
  Unique training indices: 64000
  Unique validation indices: 10178
First training sample: {'index': 1413484, 'text': 'Once upon a time, there was a sweet little dog named Spot. Spot loved to play with his ball. One day, he saw a big tree in the park. Spot had an urge to play near the tree. He did not know why, but he felt like something fun would happen there.\n\nSpot ran to the tree and started to play with his ball. He saw a pretty bird up in the tree. He wanted to be friends with the bird. Spot tried to point at the bird with his paw, but the bird did not see him. Spot felt sad, but he did not give up. He knew there was a way to make the bird see him.\n\nThe next day, Spot went back to the tree. He had a plan. He found a long stick and held it in his mouth. Spot used the stick to point at the bird. This time, the bird saw him! The bird flew down and played with Spot and his ball. They beca

wandb: 
wandb: 🚀 View run gpt-tinystories-8M-1029_165229 at: 
